In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "interim" / "cleaned.parquet"
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LABEL = "Attack Type"

print("Dataset exists:", DATA_PATH.exists())
print("Figures directory:", FIGURES_DIR)


In [ ]:

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Total classes:", df[LABEL].nunique())

display(df.head())


In [ ]:

class_counts = df[LABEL].value_counts()
class_percentages = df[LABEL].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(4)
})

display(class_summary)

print("Majority class:", class_counts.idxmax())
print("Minority class:", class_counts.idxmin())
print("Imbalance ratio:", round(class_counts.max() / class_counts.min(), 2))


In [ ]:

plt.figure(figsize=(12, 6))

sns.barplot(
    x=class_counts.values,
    y=class_counts.index,
    color="steelblue"
)

plt.title("CICIDS2017 Traffic Class Distribution")
plt.xlabel("Number of Records")
plt.ylabel("Attack Type")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

plt.figure(figsize=(12, 6))

sns.barplot(
    x=class_counts.values,
    y=class_counts.index,
    color="teal"
)

plt.xscale("log")
plt.title("CICIDS2017 Class Distribution (Logarithmic Scale)")
plt.xlabel("Number of Records (Log Scale)")
plt.ylabel("Attack Type")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "class_distribution_log.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

print("Total numerical features:", len(numeric_features))

feature_summary = df[numeric_features].describe().T

display(feature_summary.head(15))

print("\nFeatures with highest standard deviation:")
display(
    feature_summary.sort_values(
        "std", ascending=False
    ).head(10)
)


In [ ]:

corr_matrix = df[numeric_features].corr()

plt.figure(figsize=(16, 13))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("CICIDS2017 Feature Correlation Heatmap")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "feature_correlation_heatmap.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


In [ ]:

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Correlation"].abs() >= 0.90
].copy()

high_corr_pairs["Absolute Correlation"] = (
    high_corr_pairs["Correlation"].abs()
)

high_corr_pairs = high_corr_pairs.sort_values(
    "Absolute Correlation",
    ascending=False
)

print("Highly correlated feature pairs:", len(high_corr_pairs))

display(high_corr_pairs.head(20))


In [ ]:

sample_df = df.sample(
    n=min(10000, len(df)),
    random_state=42
)

selected_features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feature in zip(axes.flat, selected_features):
    sns.boxplot(
        x=sample_df[feature],
        ax=ax,
        color="steelblue"
    )
    ax.set_title(feature)
    ax.set_xlabel("Feature Value")

plt.suptitle("CICIDS2017 Feature Outlier Analysis")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "feature_outliers.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

selected_features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets"
]

outlier_results = []

for feature in selected_features:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[feature] < lower) | (df[feature] > upper)).sum()

    outlier_results.append({
        "Feature": feature,
        "Outlier Count": int(count),
        "Outlier Percentage": round(count / len(df) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary)


In [ ]:

negative_counts = (df[numeric_features] < 0).sum()

negative_summary = pd.DataFrame({
    "Feature": negative_counts.index,
    "Negative Count": negative_counts.values
})

negative_summary = negative_summary[
    negative_summary["Negative Count"] > 0
].sort_values("Negative Count", ascending=False)

negative_summary["Percentage"] = (
    negative_summary["Negative Count"] / len(df) * 100
).round(4)

display(negative_summary)


In [ ]:

figures = sorted(FIGURES_DIR.glob("*.png"))

print("Total PNG figures:", len(figures))

for figure in figures:
    print(figure.name)
